In [2]:
import torch
import torchvision
from torchvision import datasets, transforms
from pathlib import Path

# Create a common data directory
DATA_DIR = Path("./data")
DATA_DIR.mkdir(exist_ok=True)

# Download CIFAR-10 for Question 1
cifar_train = datasets.CIFAR10(
    root=str(DATA_DIR / "cifar10"),
    train=True,
    download=True
)

cifar_test = datasets.CIFAR10(
    root=str(DATA_DIR / "cifar10"),
    train=False,
    download=True
)


100.0%


In [3]:

# Download MNIST for Question 2
mnist_train = datasets.MNIST(
    root=str(DATA_DIR / "mnist"),
    train=True,
    download=True
)

mnist_test = datasets.MNIST(
    root=str(DATA_DIR / "mnist"),
    train=False,
    download=True
)

print("CIFAR-10 training images:", len(cifar_train))
print("CIFAR-10 test images:", len(cifar_test))
print("MNIST training images:", len(mnist_train))
print("MNIST test images:", len(mnist_test))

100.0%
100.0%
100.0%
100.0%

CIFAR-10 training images: 50000
CIFAR-10 test images: 10000
MNIST training images: 60000
MNIST test images: 10000


In [4]:
import urllib.request
import zipfile

# Dataset download URL
url = "https://download.pytorch.org/tutorial/hymenoptera_data.zip"

# Dataset destination
hymenoptera_dir = DATA_DIR / "hymenoptera_data"
zip_path = DATA_DIR / "hymenoptera_data.zip"

if not hymenoptera_dir.exists():
    print("Downloading Hymenoptera dataset...")

    urllib.request.urlretrieve(url, zip_path)

    print("Extracting dataset...")

    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(DATA_DIR)

    zip_path.unlink()  # Remove ZIP after extraction

    print("Hymenoptera dataset downloaded successfully!")
else:
    print("Hymenoptera dataset already exists.")

# Verify the folder structure
for split in ["train", "val"]:
    split_path = hymenoptera_dir / split
    print(f"\n{split.upper()}")

    for class_name in ["ants", "bees"]:
        class_path = split_path / class_name
        count = len(list(class_path.glob("*"))) if class_path.exists() else 0
        print(f"{class_name}: {count} files")

Extracting dataset...
Hymenoptera dataset downloaded successfully!

TRAIN
ants: 124 files
bees: 121 files

VAL
ants: 70 files
bees: 83 files


In [8]:

import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np

from torch.utils.data import DataLoader

# Use the NVIDIA GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

# Image transformations
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5),
                         (0.5, 0.5, 0.5))
])

# Load the existing CIFAR-10 datasets
trainset = torchvision.datasets.CIFAR10(
    root="./data/cifar10",
    train=True,
    download=False,
    transform=transform
)

testset = torchvision.datasets.CIFAR10(
    root="./data/cifar10",
    train=False,
    download=False,
    transform=transform
)

# Create data loaders
batch_size = 50

trainloader = DataLoader(
    trainset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,
    pin_memory=(device.type == "cuda")
)

testloader = DataLoader(
    testset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
    pin_memory=(device.type == "cuda")
)

classes = (
    "plane", "car", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
)

print("Training images:", len(trainset))
print("Test images:", len(testset))
print("Number of batches per epoch:", len(trainloader))

# Verify image and label dimensions
images, labels = next(iter(trainloader))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)


Using device: cuda
GPU: NVIDIA GeForce RTX 4060 Laptop GPU
Training images: 50000
Test images: 10000
Number of batches per epoch: 1000
Image batch shape: torch.Size([50, 3, 32, 32])
Label batch shape: torch.Size([50])


In [9]:

# Network dimensions
Din = 3 * 32 * 32
H = 100
K = 10

# Reproducibility
torch.manual_seed(42)

# Initialize parameters
# Xavier initialization for the two weight matrices
W1 = torch.randn(Din, H, device=device) * np.sqrt(2.0 / (Din + H))
b1 = torch.zeros(H, device=device)

W2 = torch.randn(H, K, device=device) * np.sqrt(2.0 / (H + K))
b2 = torch.zeros(K, device=device)

print("W1:", W1.shape)
print("b1:", b1.shape)
print("W2:", W2.shape)
print("b2:", b2.shape)


W1: torch.Size([3072, 100])
b1: torch.Size([100])
W2: torch.Size([100, 10])
b2: torch.Size([10])


In [10]:

def sigmoid(x):
    return 1.0 / (1.0 + torch.exp(-x))


def forward_pass(X, W1, b1, W2, b2):
    # Hidden layer
    Z1 = X @ W1 + b1
    A1 = sigmoid(Z1)

    # Output layer (logits)
    Z2 = A1 @ W2 + b2

    return Z1, A1, Z2


def cross_entropy_loss(logits, labels):
    # Stable cross-entropy calculation
    log_probs = torch.log_softmax(logits, dim=1)
    correct_log_probs = log_probs[
        torch.arange(labels.shape[0], device=labels.device),
        labels
    ]
    return -correct_log_probs.mean()


# Test the forward pass on one batch
X, y = next(iter(trainloader))

X = X.to(device).reshape(X.shape[0], -1)
y = y.to(device)

Z1, A1, Z2 = forward_pass(X, W1, b1, W2, b2)
loss = cross_entropy_loss(Z2, y)

print("Input shape:", X.shape)
print("Hidden activation shape:", A1.shape)
print("Output logits shape:", Z2.shape)
print("Initial cross-entropy loss:", loss.item())


Input shape: torch.Size([50, 3072])
Hidden activation shape: torch.Size([50, 100])
Output logits shape: torch.Size([50, 10])
Initial cross-entropy loss: 2.5890116691589355


In [11]:

def manual_backprop(X, y, Z1, A1, Z2, W2):
    N = X.shape[0]

    # Output gradient: softmax - one-hot labels
    probabilities = torch.softmax(Z2, dim=1)

    y_onehot = torch.zeros_like(probabilities)
    y_onehot.scatter_(1, y.unsqueeze(1), 1.0)

    dZ2 = (probabilities - y_onehot) / N

    # Output layer gradients
    dW2 = A1.T @ dZ2
    db2 = dZ2.sum(dim=0)

    # Hidden layer gradients
    dA1 = dZ2 @ W2.T
    dZ1 = dA1 * A1 * (1.0 - A1)

    # First layer gradients
    dW1 = X.T @ dZ1
    db1 = dZ1.sum(dim=0)

    return dW1, db1, dW2, db2


In [12]:

# Training hyperparameters
epochs = 10
learning_rate = 0.01
lr_decay = 0.9

# Track metrics
train_losses = []
train_accuracies = []
test_accuracies = []

for epoch in range(epochs):
    running_loss = 0.0
    correct_train = 0
    total_train = 0

    # Training mode: manually update parameters
    for images, labels in trainloader:
        # Move data to GPU and flatten images
        X = images.to(device).reshape(images.shape[0], -1)
        y = labels.to(device)

        N = X.shape[0]

        # Forward pass
        Z1, A1, Z2 = forward_pass(X, W1, b1, W2, b2)

        # Cross-entropy loss
        loss = cross_entropy_loss(Z2, y)

        # Manual backpropagation
        dW1, db1, dW2, db2 = manual_backprop(
            X, y, Z1, A1, Z2, W2
        )

        # Update parameters without autograd
        with torch.no_grad():
            W1 -= learning_rate * dW1
            b1 -= learning_rate * db1
            W2 -= learning_rate * dW2
            b2 -= learning_rate * db2

        # Accumulate loss and training accuracy
        running_loss += loss.item() * N
        predictions = Z2.argmax(dim=1)
        correct_train += (predictions == y).sum().item()
        total_train += N

    epoch_loss = running_loss / total_train
    train_acc = 100.0 * correct_train / total_train

    # Evaluate on the test set
    correct_test = 0
    total_test = 0

    with torch.no_grad():
        for images, labels in testloader:
            X = images.to(device).reshape(images.shape[0], -1)
            y = labels.to(device)

            _, _, logits = forward_pass(X, W1, b1, W2, b2)

            predictions = logits.argmax(dim=1)
            correct_test += (predictions == y).sum().item()
            total_test += y.size(0)

    test_acc = 100.0 * correct_test / total_test

    train_losses.append(epoch_loss)
    train_accuracies.append(train_acc)
    test_accuracies.append(test_acc)

    print(
        f"Epoch [{epoch + 1:02d}/{epochs}] | "
        f"Loss: {epoch_loss:.4f} | "
        f"Train accuracy: {train_acc:.2f}% | "
        f"Test accuracy: {test_acc:.2f}%"
    )

    # Learning-rate decay
    learning_rate *= lr_decay


Epoch [01/10] | Loss: 1.9652 | Train accuracy: 32.03% | Test accuracy: 36.34%
Epoch [02/10] | Loss: 1.8273 | Train accuracy: 37.23% | Test accuracy: 37.82%
Epoch [03/10] | Loss: 1.7854 | Train accuracy: 38.54% | Test accuracy: 38.50%
Epoch [04/10] | Loss: 1.7602 | Train accuracy: 39.51% | Test accuracy: 39.14%
Epoch [05/10] | Loss: 1.7425 | Train accuracy: 40.17% | Test accuracy: 39.78%
Epoch [06/10] | Loss: 1.7296 | Train accuracy: 40.61% | Test accuracy: 40.20%
Epoch [07/10] | Loss: 1.7189 | Train accuracy: 40.98% | Test accuracy: 40.16%
Epoch [08/10] | Loss: 1.7099 | Train accuracy: 41.39% | Test accuracy: 40.70%
Epoch [09/10] | Loss: 1.7029 | Train accuracy: 41.60% | Test accuracy: 40.81%
Epoch [10/10] | Loss: 1.6966 | Train accuracy: 41.90% | Test accuracy: 41.13%
